# Notebook 02: ABDM FHIR R4 Bundle Parsing & ML Dataset Reconstruction

## Objective
In compliance with the **Ayushman Bharat Digital Mission (ABDM)** guidelines, healthcare data is transferred across hospital networks as **FHIR R4 JSON Bundles**.

This notebook demonstrates:
1. Parsing complex FHIR `Patient`, `Observation`, and `Condition` resource bundles.
2. Reconstructing a structured tabular DataFrame for machine learning.
3. Validating ABHA ID patterns and verifying zero data loss.



In [ ]:
import json
import re
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

fhir_bundle_path = "../data/generated/synthetic_bundle.json"
with open(fhir_bundle_path, 'r', encoding='utf-8') as f:
    bundle = json.load(f)

print(f"Loaded FHIR Bundle containing {len(bundle.get('entry', []))} total resources.")


: 

## 1. Define & Run FHIR Parser Engine


In [ ]:
def parse_fhir_bundle(bundle_json: dict) -> pd.DataFrame:
    entries = bundle_json.get("entry", [])
    
    patients = {}
    observations = {}
    conditions = {}
    
    for entry in entries:
        res = entry.get("resource", {})
        res_type = res.get("resourceType")
        
        if res_type == "Patient":
            p_id = res.get("id")
            abha = res.get("identifier", [{}])[0].get("value", "")
            gender = res.get("gender", "")
            birth_year = int(res.get("birthDate", "1970-01-01").split("-")[0])
            age = 2026 - birth_year
            patients[p_id] = {
                "patient_id": p_id,
                "abha_id": abha,
                "gender": gender,
                "age": age
            }
        elif res_type == "Observation":
            subject_ref = res.get("subject", {}).get("reference", "")
            p_id = subject_ref.replace("Patient/", "")
            obs_text = res.get("code", {}).get("text", "")
            val = res.get("valueQuantity", {}).get("value", 0.0)
            
            if p_id not in observations:
                observations[p_id] = {}
            observations[p_id][obs_text] = val
            
        elif res_type == "Condition":
            subject_ref = res.get("subject", {}).get("reference", "")
            p_id = subject_ref.replace("Patient/", "")
            cond_text = res.get("code", {}).get("text", "")
            
            if p_id not in conditions:
                conditions[p_id] = set()
            conditions[p_id].add(cond_text)
            
    rows = []
    obs_map = {
        "Systolic Blood Pressure": "systolic_bp",
        "Diastolic Blood Pressure": "diastolic_bp",
        "Heart Rate": "heart_rate",
        "BMI": "bmi",
        "Glucose": "glucose",
        "HbA1c": "hba1c",
        "Cholesterol": "cholesterol",
        "Creatinine": "creatinine"
    }
    
    for p_id, p_info in patients.items():
        row = dict(p_info)
        p_obs = observations.get(p_id, {})
        for fhir_obs, col_name in obs_map.items():
            row[col_name] = p_obs.get(fhir_obs, np.nan)
            
        p_conds = conditions.get(p_id, set())
        row["diabetes"] = 1 if "Diabetes Mellitus" in p_conds else 0
        row["heart_disease"] = 1 if "Heart Disease" in p_conds else 0
        row["hypertension"] = 1 if "Hypertension" in p_conds else 0
        row["kidney_disease"] = 1 if "Kidney Disease" in p_conds else 0
        
        rows.append(row)
        
    return pd.DataFrame(rows)

df_parsed = parse_fhir_bundle(bundle)
print(f"Successfully reconstructed {len(df_parsed)} patient records from FHIR bundle.")
df_parsed.head()


## 2. Validate ABHA Identifiers & Data Sanity


In [ ]:
abha_pattern = re.compile(r"^91-\d{4}-\d{4}-\d{4}$")
valid_abhas = df_parsed['abha_id'].apply(lambda x: bool(abha_pattern.match(str(x))))

print(f"ABHA ID Format Validation: {valid_abhas.sum()} / {len(df_parsed)} valid ({valid_abhas.mean()*100:.1f}%)")
assert valid_abhas.all(), "Some ABHA IDs do not comply with the ABDM standard!"
print("All ABHA IDs strictly match ABDM specifications.")


## 3. Resource Breakdown Visualization


In [ ]:
resource_counts = {}
for entry in bundle.get("entry", []):
    r_type = entry.get("resource", {}).get("resourceType")
    resource_counts[r_type] = resource_counts.get(r_type, 0) + 1

plt.figure(figsize=(7, 4))
sns.barplot(x=list(resource_counts.keys()), y=list(resource_counts.values()), palette='crest')
plt.title("FHIR R4 Bundle Resource Breakdown")
plt.ylabel("Resource Count")
plt.show()
